# **Data Modelling and Evaluating**
---

## Objectives

* Answer business requirement 2: 
    * The client wants the system to identify which animal species is present in an image.


## Inputs

* inputs/datasets/animals/image/train/
* inputs/datasets/animals/image/test/
* inputs/datasets/animals/image/validation/
* Image shape embeddings: outputs/v1/image_shape.pkl

## Outputs

* Images distribution plot in train, validation, and test set.
* Image augmentation pipeline.
* Class indices mapping.
* Machine learning model creation and training.
* Saved trained model.
* Learning curve plots for model performance.
* Model evaluation stored in pickle file.
* Prediction on a random image file.


## Additional Comments | Insights | Conclusions

* N/A for now

---

## Import Packages

In [ ]:
import os
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from matplotlib.image import imread
sns.set_style("white")

## Set Seeds

In [ ]:
os.environ["PYTHONHASHSEED"] = "0"
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)

## Set Working Directory

In [ ]:
cwd = os.getcwd()
os.chdir('/workspaces/Animal_detection_camera')
print("You set a new current directory")

work_dir = os.getcwd()
work_dir

## Set input directories
Set train, validation and test paths

In [ ]:
my_data_dir = 'inputs/datasets/animals/image'
train_path = my_data_dir + '/train'
val_path = my_data_dir + '/validation'
test_path = my_data_dir + '/test'


## Set output directory

In [ ]:
version = 'v1'
file_path = f'outputs/{version}'
os.makedirs(file_path, exist_ok=True)

if 'outputs' in os.listdir(work_dir) and version in os.listdir(work_dir + '/outputs'):
    print('Old version is already available, create a new version.')
else:
    os.makedirs(name=file_path, exist_ok=True)

## Set labels

In [ ]:
labels = os.listdir(train_path)

print(
    f"Project Labels: {labels}"
)

## Set Image Shape

In [ ]:
image_shape = joblib.load(filename=f"outputs/{version}/image_shape.pkl")
image_shape = (128, 128, 3)
print("Using image shape:", image_shape)

---

# Number of images in train, test and validation data
---

In [ ]:
data = {'Set': [], 'Label': [], 'Frequency': []}
folders = ['train', 'validation', 'test']

for folder in folders:
    for label in labels:
        n = len(os.listdir(my_data_dir + '/' + folder + '/' + label))
        data['Set'].append(folder)
        data['Label'].append(label)
        data['Frequency'].append(n)
        print(f"* {folder} - {label}: {n} images")

df_freq = pd.DataFrame(data)

fig, axes = plt.subplots(3, 1, figsize=(18, 20), sharey=True)

sets = ['train', 'validation', 'test']

for i, subset in enumerate(sets):
    sns.barplot(
        data=df_freq[df_freq['Set'] == subset],
        x='Label', y='Frequency', hue='Label', dodge=False, ax=axes[i]
    )
    axes[i].set_title(f"{subset.capitalize()} Set", fontsize=16)
    axes[i].set_xlabel("Species")
    axes[i].set_ylabel("Frequency")
    axes[i].tick_params(axis='x', rotation=90)

handles, labels_legend = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_legend, loc='center', ncol=6, fontsize=10)

plt.subplots_adjust(hspace=0.4, bottom=0.1)

fig.suptitle("Image Distribution per Set", fontsize=20)
plt.savefig(f'{file_path}/labels_distribution_rows.png', dpi=150, bbox_inches='tight')
plt.show()


---

---

# Model creation
---

## ML model

Import model packages

In [ ]:
from keras.applications import MobileNetV2
from keras.models import Model
from keras.layers import Dense, Dropout, GlobalAveragePooling2D

### Model

In [ ]:
def create_tf_model():
    # Load pretrained base model
    base_model = MobileNetV2(
        input_shape=image_shape,
        include_top=False,
        weights='imagenet'
    )

    # Freeze base layers
    base_model.trainable = False

    # Add custom classification layers on top
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(len(labels), activation='softmax')(x)

    # Build the model
    model = Model(inputs=base_model.input, outputs=outputs)

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


### Model Summary

In [ ]:
create_tf_model().summary()

## Early Stopping 

In [ ]:
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

ckpt = ModelCheckpoint(
    filepath=f"{file_path}/model_epoch{{epoch:02d}}-val_acc{{val_accuracy:.2f}}.weights.h5",
    save_weights_only=True,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

## Fit model for model training

In [ ]:
model = create_tf_model()
history = model.fit(
    train_set,
    epochs=25,
    validation_data=validation_set,
    callbacks=[early_stop, reduce_lr, ckpt],
    verbose=1
)


## Save Model

In [ ]:
model.save("outputs/v1/final_model.keras")
model.save_weights('outputs/v1/animal_detector_model.weights.h5')

---

# Model Performance
---

## Model Learning Curve

In [ ]:
losses = pd.DataFrame(history.history)

sns.set_style("whitegrid")
losses[['loss', 'val_loss']].plot(style='.-')
plt.title("Loss")
plt.savefig(f'{file_path}/model_training_losses.png',
    bbox_inches='tight', dpi=150)
plt.show()

print("\n")
losses[['accuracy', 'val_accuracy']].plot(style='.-')
plt.title("Accuracy")
plt.savefig(f'{file_path}/model_training_acc.png',
    bbox_inches='tight', dpi=150)
plt.show()

## Model Evaluation

Load saved model

In [ ]:
from tensorflow.keras.models import load_model
model = create_tf_model()
model.load_weights('outputs/v1/animal_detector_model.weights.h5')

Evaluate model on test set

In [ ]:
evaluation = model.evaluate(test_set)

### Save evaluation pickle

In [ ]:
joblib.dump(value=evaluation,
    filename=f"outputs/v1/evaluation.pkl")

print("Test Evaluation:", evaluation)

# Evaluate Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


test_loss, test_acc = model.evaluate(test_set, verbose=1)
print(f"\n Test Accuracy: {test_acc:.4f}")
print(f" Test Loss: {test_loss:.4f}")

### Load true labels

In [ ]:
y_true = test_set.classes

Get predicted probablities for test set images

In [ ]:
y_pred_probs = model.predict(test_set, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

Map indices back to labels and print classification report

In [ ]:
target_names = list(train_set.class_indices.keys())

print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=target_names))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=False, cmap="inferno", 
            xticklabels=target_names, 
            yticklabels=target_names, ax=ax)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Confusion Matrix - Animal Species Classification")

Save confusion matrix

In [ ]:
save_dir = "outputs/v1"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "confusion_matrix.png")
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"Confusion matrix saved to {save_path}")

In [ ]:
plt.show()

## Predict on new data

Load a random image as PIL

In [ ]:
from tensorflow.keras.preprocessing import image
import random

# Pick a random label (species)
label = random.choice(labels)

# Pick a random image index from that label's folder
pointer = random.randint(0, len(os.listdir(test_path + '/' + label)) - 1)

# Load the image
pil_image = image.load_img(
    test_path + '/' + label + '/' + os.listdir(test_path + '/' + label)[pointer],
    target_size=image_shape[:2],
    color_mode='rgb'
)

print(f"Selected species: {label}")
print(f"Image shape: {pil_image.size}, Image mode: {pil_image.mode}")
pil_image


Convert image to array and prepare for prediction

In [ ]:
my_image = image.img_to_array(pil_image)
my_image = np.expand_dims(my_image, axis=0)/255
print(my_image.shape)
print("Image batch shape:", my_image.shape)
print("True label:", label)


Predict class probabilities

In [ ]:
# Get prediction probabilities for all classes
pred_probs = model.predict(my_image)[0]

target_map = {v: k for k, v in train_set.class_indices.items()}

pred_class_index = np.argmax(pred_probs)
pred_class = target_map[pred_class_index]

print("True label:", label)
print("Predicted label:", pred_class)
print("Prediction confidence:", pred_probs[pred_class_index])

# Show top 3 predictions and their probabilities
top_3 = np.argsort(pred_probs)[-3:][::-1]
print("\nTop 3 predictions:")
for i in top_3:
    print(f"{target_map[i]}: {pred_probs[i]:.4f}")

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "filepath": test_set.filepaths,
    "true_label": [target_names[i] for i in y_true],
    "predicted_label": [target_names[i] for i in y_pred]
})

results_df.to_csv("inputs/datasets/animal_predictions.csv", index=False)

results_df["image"] = test_set.filenames
